In [6]:
import re
import pandas as pd

def parse_triton_log(path):
    rows = []

    # State variables
    block_e = None
    block_n = None
    batch_size = None
    seq_len = None
    hidden_dim = None

    # Temporary timing storage
    timings = {}

    with open(path, "r") as f:
        for line in f:
            line = line.strip()

            # Kernel params
            m = re.search(r"BLOCK_E=(\d+), BLOCK_N=(\d+)", line)
            if m:
                block_e = int(m.group(1))
                block_n = int(m.group(2))
                continue

            # Batch shape
            m = re.search(r"torch.Size\(\[(\d+), (\d+), (\d+)\]\)", line)
            if m:
                batch_size = int(m.group(1))
                seq_len = int(m.group(2))
                hidden_dim = int(m.group(3))
                timings = {}
                continue

            # Timings
            patterns = {
                "normal_decode_ms": r"normal_decode decode tim:\s+([\d.]+)",
                "dense_bmm_ms": r"Normal BMM \(dense\):\s+([\d.]+)",
                "cusparse_decode_ms": r"naive_cusparse_decode decode tim:\s+([\d.]+)",
                "cusparse_kernel_ms": r"Naive cusparse decoder time:\s+([\d.]+)",
                "triton_decode_ms": r"custom_triton_decode decode tim:\s+([\d.]+)",
                "triton_kernel_ms": r"Custom triton kernel time:\s+([\d.]+)",
            }

            for key, pat in patterns.items():
                m = re.search(pat, line)
                if m:
                    timings[key] = float(m.group(1))

            # Once we have all timings, commit a row
            if len(timings) == 6:
                rows.append({
                    "BLOCK_E": block_e,
                    "BLOCK_N": block_n,
                    "batch_size": batch_size,
                    "seq_len": seq_len,
                    "hidden_dim": hidden_dim,
                    **timings
                })
                timings = {}

    return pd.DataFrame(rows)

data = parse_triton_log("37976203_triton_testing.out")
print(data.head(5))

   BLOCK_E  BLOCK_N  batch_size  seq_len  hidden_dim  normal_decode_ms  \
0       32       32           1      128         768          0.685056   
1       32       32           4      128         768          1.832000   
2       32       32           8      128         768          3.353600   
3       32       32          16      128         768          6.250496   
4       32       32          32      128         768         12.378112   

   dense_bmm_ms  cusparse_decode_ms  cusparse_kernel_ms  triton_decode_ms  \
0      4248.576          168.939514          171212.799       6397.189941   
1      2787.328            8.007680            9096.192       4439.115723   
2      4208.640            4.753408            5551.104          8.236032   
3      7033.856            9.231360           10028.032         17.389568   
4     13328.384           18.163712           19106.815         31.980543   

   triton_kernel_ms  
0       6399177.734  
1       4440139.648  
2          9032.704  
3   

In [9]:
print(data[(data["BLOCK_E"] == 32) & (data["BLOCK_N"] == 256)])
data.to_csv("parsed_custom_kernel_grid_search.csv")

    BLOCK_E  BLOCK_N  batch_size  seq_len  hidden_dim  normal_decode_ms  \
16       32      256           1      128         768          0.694272   
17       32      256           4      128         768          1.835008   
18       32      256           8      128         768          3.350528   
19       32      256          16      128         768          6.258688   
20       32      256          32      128         768         12.369920   
21       32      256          64      128         768         24.633345   
22       32      256         128      128         768         47.534111   
23       32      256         256      128         768         95.478783   

    dense_bmm_ms  cusparse_decode_ms  cusparse_kernel_ms  triton_decode_ms  \
16      1969.152          171.413498          172749.817      10351.474609   
17      2722.848            7.694272            8536.064       6196.927246   
18      4128.704            4.782080            5702.656          1.136640   
19      6992

In [27]:
data['speedup'] = data['dense_bmm_ms'] / data['triton_kernel_ms']

for batch_size in [1, 4, 8, 16, 32, 64, 128, 256]:
    print(f"##### Best Params for batch size {batch_size} #####")
    batch_data = data[data["batch_size"] == batch_size]
    max_val = batch_data["triton_kernel_ms"].min()
    print(batch_data[batch_data["triton_kernel_ms"] == max_val][['batch_size', 'BLOCK_E', 'BLOCK_N']])
    print(batch_data[batch_data["triton_kernel_ms"] == max_val][['dense_bmm_ms', 'cusparse_kernel_ms', 'triton_kernel_ms', 'speedup']])

##### Best Params for batch size 1 #####
   batch_size  BLOCK_E  BLOCK_N
0           1       32       32
   dense_bmm_ms  cusparse_kernel_ms  triton_kernel_ms   speedup
0      4248.576          171212.799       6399177.734  0.000664
##### Best Params for batch size 4 #####
   batch_size  BLOCK_E  BLOCK_N
9           4       32      128
   dense_bmm_ms  cusparse_kernel_ms  triton_kernel_ms   speedup
9      2577.408            8879.104       3052291.016  0.000844
##### Best Params for batch size 8 #####
    batch_size  BLOCK_E  BLOCK_N
18           8       32      256
    dense_bmm_ms  cusparse_kernel_ms  triton_kernel_ms   speedup
18      4128.704            5702.656          1778.688  2.321208
##### Best Params for batch size 16 #####
    batch_size  BLOCK_E  BLOCK_N
19          16       32      256
    dense_bmm_ms  cusparse_kernel_ms  triton_kernel_ms   speedup
19      6992.896           10170.368          2627.584  2.661341
##### Best Params for batch size 32 #####
    batch_size  B